# MAI-Voice-2

Direct expressive, multilingual, and long-form speech synthesis.

| | |
|---|---|
| Released | 2026-06-02 |
| Publisher | Microsoft AI |
| Capability | [Audio / Speech](../../3-primers/audio-speech.md) |
| Model card | [MAI-Voice-2](https://ai.azure.com/catalog/models/MAI-Voice-2) |
| Announcement | [Release post](https://microsoft.ai/news/mai-voice-2/) |
| Pricing | Speech synthesis is billed per character; rates vary by region and tier — [source](https://azure.microsoft.com/pricing/details/speech/) |

**Notebooks in this capsule**

- [mai-voice-2-01-direct-expressive-delivery.ipynb](mai-voice-2-01-direct-expressive-delivery.ipynb)
- [mai-voice-2-02-test-multilingual-long-form.ipynb](mai-voice-2-02-test-multilingual-long-form.ipynb)

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

## About this release

MAI-Voice-2 is a text-to-speech model for expressive, multilingual, and long-form audio.
These notebooks turn the same text into controlled listening comparisons so you can judge
delivery, locale handling, and consistency for your own scenario.

> **Public preview.** MAI-Voice is available through Azure Speech in public preview, without
> a service-level agreement, and isn't recommended for production workloads.

### Before You Begin

| Detail | Value |
|---|---|
| Retirement date | No retirement date announced |
| Regions | Check the live [Azure Speech region table](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions) before creating the resource |

Create an Azure Speech resource in a region that supports MAI models.

**Required environment variables:**

```text
MICROSOFT_FOUNDRY_ENDPOINT   # set by setup.ipynb
MICROSOFT_FOUNDRY_API_KEY    # Speech or AIServices resource key
AZURE_SPEECH_ENDPOINT        # e.g. https://<resource>.cognitiveservices.azure.com/
```

The notebooks derive the service origin from `AZURE_SPEECH_ENDPOINT` and never embed a key.
The endpoint and key must belong to the same resource and region.

### What you'll learn

- Synthesize text with the Azure Speech SDK and a prebuilt MAI-Voice-2 voice.
- Hold text and voice constant while comparing SSML styles and intensity.
- Test multilingual, code-switched, and chunked long-form scripts with repeatable inputs.
- Save WAV outputs and review listening evidence without treating subjective impressions as metrics.

### Choose a voice and delivery

The supplied SDK example uses `en-US-Ethan:MAI-Voice-2`. MAI-Voice-2 also provides
locale-specific prebuilt voices. A voice's locale and supported styles determine which SSML
experiments are valid; use the live [prebuilt voice table](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-voices#prebuilt-voices)
as the source of truth.

Expressive synthesis uses `mstts:express-as`:

```xml
<mstts:express-as style="hopeful" styledegree="1.2">
  We found a safe route through the storm.
</mstts:express-as>
```

Use plain text synthesis for a baseline and SSML synthesis for controlled style comparisons.
The notebooks request PCM WAV output so Python can inspect durations and join compatible
segments without another audio dependency.

### Know the boundaries

- MAI-Voice-2 favors fidelity and long-form consistency over latency-critical interaction.
  Compare it with MAI-Voice-2-Flash when responsiveness is the primary constraint.
- Style availability varies by voice. Unsupported style and voice combinations can be
  rejected or produce an unhelpful comparison.
- Listening tests are qualitative. Keep script, voice, format, and playback conditions fixed
  before attributing a difference to style or locale.
- Instant voice cloning is a gated feature. It requires limited-access approval, licensed or
  consented voice material, and system-level consent safeguards. It is intentionally not
  included in executable notebook cells.

## 1. Before You Begin

Use this notebook to compare MAI-Voice-2 deliveries while holding the script and voice constant. MAI-Voice is in public preview and is not recommended for production workloads.

| Detail | Value |
|---|---|
| Released | 2026-06-02 |
| Model card | [MAI-Voice-2](https://ai.azure.com/catalog/models/MAI-Voice-2) |
| Pricing | Billed per character; check [current Speech pricing](https://azure.microsoft.com/pricing/details/speech/) |
| Setup | Complete [setup.ipynb](../../setup.ipynb), then create a supported Speech resource |

The examples use the prebuilt `en-US-Ethan:MAI-Voice-2` voice and save PCM WAV files under `output/`. Use headphones and a consistent playback volume for comparisons.

In [ ]:
## 2. Verify your environment
%pip install azure-cognitiveservices-speech python-dotenv --quiet

import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

load_dotenv()
required = ["MICROSOFT_FOUNDRY_ENDPOINT", "MICROSOFT_FOUNDRY_API_KEY", "AZURE_SPEECH_ENDPOINT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Complete setup.ipynb and add AZURE_SPEECH_ENDPOINT before continuing."
    )

parsed = urlparse(os.environ["AZURE_SPEECH_ENDPOINT"])
if parsed.scheme != "https" or not parsed.netloc:
    raise ValueError("AZURE_SPEECH_ENDPOINT must be an HTTPS resource endpoint.")

SPEECH_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
SPEECH_KEY = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Environment ready; audio will be written to {OUTPUT_DIR.resolve()}")

## 3. Establish a baseline

Start with plain-text synthesis. This gives every later SSML result a neutral reference with the same text, voice, endpoint, and WAV format. The helper surfaces cancellation details instead of returning an empty or partial file.

In [ ]:
## 4. Synthesize and inspect the baseline
import html
import time
import wave

import azure.cognitiveservices.speech as speechsdk
from IPython.display import Audio, display

DEFAULT_VOICE = "en-US-Ethan:MAI-Voice-2"
DEFAULT_LOCALE = "en-US"

def build_ssml(text, *, voice, locale, style=None, style_degree=None):
    safe_text = html.escape(text)
    if style is None:
        body = safe_text
    else:
        degree = "" if style_degree is None else f' styledegree="{style_degree}"'
        body = f'<mstts:express-as style="{style}"{degree}>{safe_text}</mstts:express-as>'
    return (
        f'<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" '
        f'xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="{locale}">'
        f'<voice name="{voice}">{body}</voice></speak>'
    )

def wav_duration(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getnframes() / audio.getframerate()

def synthesize(text, filename, *, voice=DEFAULT_VOICE, locale=DEFAULT_LOCALE, style=None, style_degree=None):
    path = OUTPUT_DIR / filename
    config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, endpoint=SPEECH_ENDPOINT)
    config.speech_synthesis_voice_name = voice
    config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm)
    output = speechsdk.audio.AudioOutputConfig(filename=str(path))
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=config, audio_config=output)
    started = time.perf_counter()
    if style is None:
        result = synthesizer.speak_text_async(text).get()
    else:
        ssml = build_ssml(text, voice=voice, locale=locale, style=style, style_degree=style_degree)
        result = synthesizer.speak_ssml_async(ssml).get()
    elapsed = time.perf_counter() - started
    if result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
        details = result.cancellation_details
        raise RuntimeError(f"Synthesis canceled: {details.reason}; {details.error_details}")
    return {"path": path, "seconds": wav_duration(path), "request_seconds": elapsed}

SCRIPT = "The research team found a safe route through the storm before nightfall."
baseline = synthesize(SCRIPT, "01-baseline.wav")
print(baseline)
display(Audio(filename=str(baseline["path"])))

## 5. Compare expressive directions

Generate an A/B matrix from the exact baseline script. Ethan supports the selected styles in the official voice table. `styledegree` changes intensity; it does not change the requested emotion. Listen for pacing, emphasis, pitch movement, and whether the delivery still fits the sentence.

In [ ]:
## 6. Render a style and intensity matrix
import pandas as pd

directions = [
    ("hopeful", 0.7),
    ("hopeful", 1.4),
    ("fearful", 1.0),
    ("relieved", 1.0),
    ("whispering", 1.0),
]

rows = [{"delivery": "baseline", **baseline}]
for style, degree in directions:
    result = synthesize(
        SCRIPT,
        f"02-{style}-{str(degree).replace('.', '-')}.wav",
        style=style,
        style_degree=degree,
    )
    rows.append({"delivery": f"{style} ({degree})", **result})

comparison = pd.DataFrame(rows)
display(comparison[["delivery", "seconds", "request_seconds", "path"]])
for row in rows:
    print(row["delivery"])
    display(Audio(filename=str(row["path"])))

## 7. Direct a short audio scene

A single sentence can make a style sound convincing even when transitions fail. This miniature radio scene tests three consecutive directions with one persona. Review whether the emotional progression sounds coherent rather than judging each clip in isolation.

In [ ]:
## 8. Sequence and join the scene
scene = [
    ("whispering", 0.9, "Keep the lantern low. The bridge is just ahead."),
    ("fearful", 1.1, "Wait. Did you hear that behind us?"),
    ("relieved", 1.2, "It was only the wind. We made it across."),
]

scene_paths = []
for index, (style, degree, line) in enumerate(scene, start=1):
    result = synthesize(line, f"03-scene-{index:02d}-{style}.wav", style=style, style_degree=degree)
    scene_paths.append(result["path"])
    print(f"{style}: {line}")
    display(Audio(filename=str(result["path"])))

def join_wavs(paths, destination):
    frames = []
    expected = None
    for path in paths:
        with wave.open(str(path), "rb") as source:
            params = source.getparams()
            signature = (params.nchannels, params.sampwidth, params.framerate, params.comptype)
            if expected is None:
                expected = signature
            elif signature != expected:
                raise ValueError(f"Incompatible WAV parameters in {path}")
            frames.append(source.readframes(source.getnframes()))
    with wave.open(str(destination), "wb") as target:
        target.setnchannels(expected[0])
        target.setsampwidth(expected[1])
        target.setframerate(expected[2])
        target.setcomptype(expected[3], "not compressed")
        for chunk in frames:
            target.writeframes(chunk)
    return destination

scene_path = join_wavs(scene_paths, OUTPUT_DIR / "03-complete-scene.wav")
print(f"Complete scene: {wav_duration(scene_path):.2f} seconds")
display(Audio(filename=str(scene_path)))

## 9. Your Turn to Explore

1. Replace the factual baseline with an instruction, apology, or celebration and identify which styles remain appropriate.
2. Hold style constant and compare `styledegree` at three values that bracket your preferred delivery.
3. Redirect the scene with `en-US-Harper:MAI-Voice-2`, checking that every requested style is supported first.

## 10. Summary

You established a plain-text baseline, compared SSML style and intensity with controlled inputs, and reviewed emotional transitions in a short scene. Reach for MAI-Voice-2 when fidelity, expressiveness, or sustained narration matters more than response latency; evaluate MAI-Voice-2-Flash for interactive agents. See the [Audio / Speech primer](../../3-primers/audio-speech.md) for related terminology.

## 11. References

- [MAI-Voice overview](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-voices) — supported SSML controls, voices, styles, and preview status.
- [MAI-Voice-2 model card](https://ai.azure.com/catalog/models/MAI-Voice-2) — official model capabilities.
- [Introducing MAI-Voice-2](https://microsoft.ai/news/mai-voice-2/) — release examples and intended uses.
- [Speech synthesis with the Speech SDK](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/get-started-text-to-speech) — SDK configuration and result handling.
- [Azure Speech SDK samples](https://github.com/Azure-Samples/cognitive-services-speech-sdk) — additional Python and cross-language examples.